# Refine a protein–ligand binder complex with Rosetta

Relax a positioned complex and run the upstream interface design protocol.

This notebook previews the configuration by default. Edit your paths and set `RUN_TASK = True` when you are ready to run.


## 1. Before you start

- Install the `chemistry` extra and obtain Rosetta, its database, and DAlphaBall separately.
- Obtain the upstream MaSIF-neosurf `rosetta_scripts/seed_refine` protocols and utilities.
- Supply a target PDB containing the ligand and a binder PDB **already positioned relative to that target**. Both must use the same coordinate system.
- Provide the ligand template SDF, or set `ligand_sdf` to a prepared, positioned ligand SDF instead.
- Set `legacy_python` to a Python interpreter compatible with your Rosetta `molfile_to_params.py` script.


Use the `Python (masif_torch)` kernel. See [installation](../README.md#installation) and the [user guide](../USAGE.md).


## 2. Open the project


In [ ]:
from pathlib import Path
import importlib.util
import os
import shlex
import shutil
import subprocess
import sys
import tempfile

from omegaconf import OmegaConf

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file() and (path / "examples").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the MaSIF_torch repository or its tutorials directory.")
os.chdir(ROOT)
CLI = [sys.executable, "-m", "masif"]
RUN_TASK = False  # Change to True only after editing the paths below.
print("Repository:", ROOT)
print("Python:", sys.executable)


## 3. Choose your inputs and settings

Paths below are relative to the repository unless you provide an absolute path. Choose a separate output directory for each run.


In [ ]:
TASK = "rosetta"
config = OmegaConf.create({
    "task": "rosetta", "target": "data/input/protein_ligand.pdb", "target_chains": "A",
    "binder": "data/input/positioned_binder.pdb", "binder_chains": "B",
    "ligand": "FKA", "ligand_chain": "C", "template_sdf": "data/input/ligand_template.sdf",
    "ligand_sdf": None,
    "protocol_dir": "/path/to/masif-neosurf/rosetta_scripts/seed_refine/src",
    "binary": "/path/to/rosetta/main/source/bin/rosetta_scripts.linuxgccrelease",
    "database": "/path/to/rosetta/main/database",
    "dalphaball": "/path/to/DAlphaBall.gcc",
    "chemistry_python": sys.executable,
    "legacy_python": "/path/to/rosetta_compatible_python/bin/python",
    "output": "results/tutorial_rosetta", "seed": 42, "verify_repeat": False,
    "timeout_seconds": 7200,
})
# Match target_chains, binder_chains, ligand, and ligand_chain to your actual files.


## 4. Preview your configuration

This prints the Hydra configuration without loading your input data or running the task.


In [ ]:
config_directory = tempfile.TemporaryDirectory(prefix="masif_tutorial_")
config_path = Path(config_directory.name) / "task.yaml"
OmegaConf.save(config, config_path)
command = CLI + [TASK, "--config", str(config_path)]
print(shlex.join(command))
subprocess.run(command + ["--cfg", "job"], check=True)


## 5. Run when ready

After editing the settings, set `RUN_TASK = True` in the setup cell and rerun the notebook. The checks below help identify missing inputs before starting.


In [ ]:
if RUN_TASK:
    ligand_input = "ligand_sdf" if config.ligand_sdf else "template_sdf"
    for key in ("target", "binder", ligand_input):
        filename = config.get(key)
        if filename and not Path(filename).is_file():
            raise FileNotFoundError(f"Missing {key}: {filename}. Update the input paths.")
    for key in ("binary", "dalphaball", "legacy_python", "chemistry_python"):
        if not Path(config[key]).is_file() or not os.access(config[key], os.X_OK):
            raise FileNotFoundError(f"Set {key} to an installed executable.")
    if not Path(config.database).is_dir():
        raise FileNotFoundError("Set database to the Rosetta database directory.")
    for name in ("relax.xml", "interface_design.xml", "no_ref.rosettacon2018.beta_nov16.txt"):
        if not Path(config.protocol_dir, name).is_file():
            raise FileNotFoundError(f"Missing upstream protocol {name}. Update protocol_dir.")
    if not (Path(config.protocol_dir).parent / "utils/other/get_mol_targetres.py").is_file():
        raise FileNotFoundError("Obtain the upstream seed_refine utils folder alongside src.")
    if not (Path(config.database).parent / "source/scripts/python/public/molfile_to_params.py").is_file():
        raise FileNotFoundError("The Rosetta installation must include source/scripts/python/public/molfile_to_params.py.")
    subprocess.run([config.chemistry_python, "-c", "import rdkit, Bio"], check=True)
    subprocess.run(command, check=True)
else:
    print("Preview only. Edit your paths and set RUN_TASK = True to run.")


## 6. Next steps

Find the refined structures under your chosen `output` directory. To start with another target, see [ligand preparation](prepare_ligand.ipynb); for docking candidates, see [seed docking](seed.ipynb).
